# Homework: Interventions and circuits in GPT-2

Use GPT-2 small (124M). CPU works.

In [ ]:
%pip install -q "torch>=2.6,<3" "transformers>=4.48,<5"

In [ ]:
import os
import torch
from transformers import AutoTokenizer, GPT2LMHeadModel

device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "openai-community/gpt2"
revision = "607a30d783dfa663caf39e06633721c8d4cfcd7e"

model = GPT2LMHeadModel.from_pretrained(model_id, revision=revision, attn_implementation="eager").to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
model.eval().requires_grad_(False)

In [ ]:
sites = [*model.transformer.h, model.transformer.ln_f]
name_ids = [tokenizer.encode(name, add_special_tokens=False) for name in (" Mary", " John")]
assert all(len(ids) == 1 for ids in name_ids)
mary, john = [ids[0] for ids in name_ids]

### Data and conventions

Parts 1–2 use **16 discovery and 12 test prompts**, with both name orders and givers. Adjacent rows are partners; test rows 0/1 form the library pair.

At the last non-padding token, compare the two name logits:

$$
g=\ell_{\mathrm{Mary}}-\ell_{\mathrm{John}},\qquad
s=\begin{cases}+1&\text{Mary target},\\-1&\text{John target}.\end{cases}
$$

**Site $k$** is after $k$ complete blocks: site 0 is before block 0, and site 12 is before final normalization. Indices start at zero.

In [ ]:
discovery_templates = [
    "When {a} and {b} went to the store, {giver} gave a book to",
    "After {a} and {b} met downtown, {giver} handed a letter to",
    "While {a} and {b} were at the park, {giver} tossed a ball to",
    "Because {a} and {b} were friends, {giver} sent a gift to",
]
test_templates = [
    "At the library {a} and {b} chose some books. Afterwards {giver} handed a novel to",
    "When {a} and {b} arrived at the office, {giver} passed a folder to",
    "During the picnic {a} and {b} unpacked the basket. Then {giver} offered an apple to",
]

def make_batch(templates):
    variants = [(t.format(a=a, b=b, giver=giver), sign) for t in templates
                for a, b in [("Mary", "John"), ("John", "Mary")]
                for giver, sign in [("John", 1), ("Mary", -1)]]
    texts, signs = zip(*variants)
    inputs = tokenizer(list(texts), padding=True, return_tensors="pt", add_special_tokens=False).to(device)
    rows = torch.arange(len(texts), device=device)
    last, partner = inputs["attention_mask"].sum(1) - 1, rows ^ 1
    assert torch.equal(last, last[partner])
    assert ((inputs["input_ids"] != inputs["input_ids"][partner]).sum(1) == 1).all()
    return dict(texts=list(texts), inputs=inputs, signs=torch.tensor(signs, device=device),
                rows=rows, last=last, partner=partner)

discovery, test = make_batch(discovery_templates), make_batch(test_templates)
print(*test["texts"][:2], sep="\n")

`run(batch, site, edit)` returns logit gaps and pre-edit residuals `[batch, tokens, 768]`. The optional `edit` receives a clone and returns the replacement. Use `batch["last"]` for the last non-padding positions.

In [ ]:
@torch.no_grad()
def run(batch, site, edit=None):
    cache = {}
    def hook(module, args):
        residual = args[0]
        cache["residual"] = residual.detach().clone()
        if edit is not None:
            return (edit(residual.clone()), *args[1:])
    with sites[site].register_forward_pre_hook(hook):
        logits = model(**batch["inputs"], use_cache=False).logits
    final = logits[batch["rows"], batch["last"]]
    return final[:, mary] - final[:, john], cache["residual"]

## 1. Activation patching

At **site of your choice**, replace each test prompt's final-token residual with its partner's cached vector, keeping input tokens unchanged:

$$
r_i^{(?)}[t_i]\leftarrow r_{\pi(i)}^{(?)}[t_i],\qquad \pi(i)=\text{partner of }i.
$$

Show the library pair's before/after gaps in both directions. Across all 12 prompts, report mean donor-directed shift $-s_i(g_i^{\mathrm{patch}}-g_i)$ and how many prefer the donor's answer.

Check self-patching and repeat the swap at **sites 0 and 12**. Explain these controls and why a shift may not flip the answer. [Patching reference](https://arxiv.org/abs/2309.16042)

Find the best spot to patch!

In [ ]:
patch_base, patch_states = run(test, ?)
rows, last, partner = test["rows"], test["last"], test["partner"]
# TODO: define swap(residual), then call run(test, ?, edit=swap).

## 2. Activation steering

At **site of your choice**, estimate a direction from discovery prompts only:

$$
d=\operatorname{mean}_{s_i=+1}r_i^{(?)}[t_i]-\operatorname{mean}_{s_i=-1}r_i^{(?)}[t_i].
$$

On test prompts, apply $r_i^{(?)}[t_i]\leftarrow r_i^{(?)}[t_i]+\alpha d$ for $\alpha\in\{-2,-1,0,1,2\}$. Keep $d$'s original magnitude. Report mean logit shift and the fraction preferring Mary for each coefficient.

At $\alpha=1$, compare five equal-norm random directions orthogonal to $d$:

$$
z\sim\mathcal N(0,I),\qquad
v=z-\frac{z^\top d}{\lVert d\rVert^2}d,\qquad
q=\lVert d\rVert\frac{v}{\lVert v\rVert}.
$$

Find the best place to steer!

In [ ]:
_, discovery_states = run(discovery, ?)
x = discovery_states[discovery["rows"], discovery["last"]]
labels = discovery["signs"]
alphas = [-2, -1, 0, 1, 2]
# TODO: compute d, then use run(test, ?, edit=...) for steering and controls.

## 3. Discover a circuit and inspect a neuron

Choose one behavior and build your own prompts:

- **Indirect-object identification:** “Mary and John … John gave a book to” → “ Mary”. Use new names or constructions. [Paper](https://arxiv.org/abs/2211.00593)
- **Greater-than:** “The war lasted from the year 1732 to the year 17”. Compare probability mass on two-digit endings above 32 versus at or below 32. Check tokenization. [Paper](https://arxiv.org/abs/2305.00586)
- **Subject–verb agreement:** “The key/keys near the cabinets” → “ is/are”. Vary subject and distractor number. [Paper](https://arxiv.org/abs/2506.22105)

**Circuit.** Define a scalar metric, clean/counterfactual pairs, and discovery/validation/test splits. Specify graph components and edges, including MLPs. Use **EAP or EAP-IG** on discovery data and select circuit size on validation data. Existing implementations are allowed; enable activation gradients even with frozen weights.

Show performance versus circuit size and one input-to-output route. On test data, compare the retained circuit with the full model, counterfactual baseline and an equally sized random circuit. Keep the clean metric fixed across interventions. Check that retaining every edge recovers the full model, and test one attributed edge directly. Report raw scores alongside normalized recovery.

**Neuron.** Select a post-GELU neuron in a retained MLP using discovery data. Give its layer, index and position; compare high/low activation contexts and state a hypothesis. On held-out prompts, ablate or replace it, with a control neuron and self-replacement. Explain its contribution to the circuit, including null results.

[EAP](https://arxiv.org/abs/2310.10348); [EAP-IG](https://arxiv.org/abs/2403.17806)